# Word Embeddings and Analogy Prediction

This notebook explores vector relationships in word embeddings using country–capital predictions and analogy tasks based on cosine similarity and euclidean distance.

## 1. Environment setup

Importing libraries and loading the resources used in the analysis.

In [ ]:
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
!git clone  https://github.com/mmazurek-wat/nlp-resources-edu.git
resource_path = '/content/nlp-resources-edu/05_Embedding_lab/'

Cloning into 'nlp-resources-edu'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 60 (delta 17), reused 32 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 20.15 MiB | 23.60 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [ ]:
sys.path.append(resource_path)
from os import path

## 2. Loading the capitals dataset

Loading the evaluation dataset containing country–capital relationships.

In [ ]:
capitals_file = 'capitals.txt'
capitals_file_path = path.join(resource_path, capitals_file)
data = pd.read_csv(capitals_file_path, delimiter=' ')
data.columns = ['city1', 'country1', 'city2', 'country2']
data.head(5)

,city1,country1,city2,country2
0,Athens,Greece,Bangkok,Thailand
1,Athens,Greece,Beijing,China
2,Athens,Greece,Berlin,Germany
3,Athens,Greece,Bern,Switzerland
4,Athens,Greece,Cairo,Egypt


## 3. Loading word embeddings

Using a preloaded subset of word embeddings for the experiments.

In [ ]:
embedding_subset_file = 'word_embeddings_subset.p'
embedding_subset_file_path = path.join(resource_path, embedding_subset_file)

word_embeddings = pickle.load(open(embedding_subset_file_path, "rb"))
len(word_embeddings)

243

In [ ]:
print(*word_embeddings.keys())
print("dimension: {}".format(word_embeddings['Spain'].shape[0]))

country city China Iraq oil town Canada London England Australia Japan Pakistan Iran gas happy Russia Afghanistan France Germany Georgia Baghdad village Spain Italy Beijing Jordan Paris Ireland Turkey Egypt Lebanon Taiwan Tokyo Nigeria Vietnam Moscow Greece Indonesia sad Syria Thailand Libya Zimbabwe Cuba Ottawa Tehran Sudan Kenya Philippines Sweden Poland Ukraine Rome Venezuela Switzerland Berlin Bangladesh Portugal Ghana Athens king Madrid Somalia Dublin Qatar Chile Islamabad Bahrain Nepal Norway Serbia Kabul continent Brussels Belgium Uganda petroleum Cairo Denmark Austria Jamaica Georgetown Bangkok Finland Peru Romania Bulgaria Hungary Vienna Kingston Manila Cyprus Azerbaijan Copenhagen Fiji Tunisia Kazakhstan queen Beirut Jakarta Croatia Belarus Algeria Malta Morocco Rwanda Bahamas Damascus Ecuador Angola Canberra Liberia Honduras Tripoli Slovakia Doha Armenia Taipei Oman Nairobi Santiago Guinea Uruguay Stockholm Slovenia Zambia Havana Uzbekistan Belgrade Mogadishu Khartoum Botswa

## 4. Similarity measures

Defining cosine similarity and euclidean distance for comparing word vectors.

In [ ]:
def cosine_similarity(A, B):
    dot = np.dot(A,B)
    norma = np.linalg.norm(A)
    normb = np.linalg.norm(B)
    cos = dot / (norma * normb)
    return cos

In [ ]:
king = word_embeddings['king']
queen = word_embeddings['queen']

similarity1 = cosine_similarity(king, queen)
print(f"king and queen: {similarity1:.3f}")

tokyo = word_embeddings['Tokyo']
japan = word_embeddings['Japan']

similarity2 = cosine_similarity(tokyo, japan)
print(f"tokyo and japan: {similarity2:.3f}")

king and queen: 0.651
tokyo and japan: 0.700


In [ ]:
def euclidean(A, B):
    d = np.linalg.norm(A-B)
    return d

In [ ]:
euclidean(king, queen)

np.float32(2.4796925)

## 5. Predicting countries from capital relationships

Using vector arithmetic to infer a country from an analogy of the form: city1 : country1 = city2 : country2.

In [ ]:
def get_country(city1, country1, city2, embeddings):

    group = set((city1, country1, city2))

    city1_emb = embeddings[city1]
    country1_emb = embeddings[country1]
    city2_emb = embeddings[city2]

    vec = country1_emb - city1_emb + city2_emb
    similarity = -1.0
    country = ('', -1.0)

    for word in embeddings.keys():
        if word not in group:
            word_emb = embeddings[word]

            cur_similarity = cosine_similarity(vec, word_emb)

            if cur_similarity > similarity:
                similarity = cur_similarity
                country = (word, cur_similarity)
    return country

In [ ]:
get_country('Athens', 'Greece', 'Cairo', word_embeddings)

('Egypt', np.float32(0.7626821))

## 6. Model evaluation

Evaluating prediction accuracy on the capitals dataset.

In [ ]:
def get_accuracy(word_embeddings, data):
    num_correct = 0

    for i, row in data.iterrows():
        city1 = row['city1']
        country1 = row['country1']
        city2 =  row['city2']
        country2 = row['country2']

        predicted_country2, _ = get_country(city1, country1, city2, word_embeddings)

        if predicted_country2 == country2:
            num_correct += 1

    m = len(data)
    accuracy = num_correct / m
    return accuracy

In [ ]:
accuracy = get_accuracy(word_embeddings, data)
print(f"Accuracy is {accuracy:.2f}")

Accuracy is 0.92


## 7. Analogy prediction with Euclidean distance

Finding the closest word to a target vector obtained from analogy-based vector arithmetic.

In [ ]:
def find_closest_word(v):
    best_word = None
    best_dist = float('inf')
    for w, emb in word_embeddings.items():
        d = np.linalg.norm(v - emb)
        if d < best_dist:
            best_dist = d
            best_word = w
    return best_word, best_dist

In [ ]:
def get_analogy(rel_src1, rel_trg1, rel_src2):
  v = word_embeddings[rel_trg1] - word_embeddings[rel_src1] + word_embeddings[rel_src2]

  skip = {rel_src1, rel_trg1, rel_src2}

  best_word, best_dist = find_closest_word(v)

  if best_word in skip:
      distances = [(np.linalg.norm(v - emb), w) for w, emb in word_embeddings.items() if w not in skip]
      best_dist, best_word = min(distances, key=lambda x: x[0])

  return best_word

In [ ]:
result = get_analogy('Athens', 'Greece', 'Tokyo')
print(f"Athens : Greece = Tokyo : {result}")

result = get_analogy('Poland', 'Warsaw', 'Germany')
print(f"Poland : Warsaw = Germany : {result}")

Athens : Greece = Tokyo : Japan
Poland : Warsaw = Germany : Berlin


## Conclusion

Word embeddings capture meaningful semantic relationships that can be used for analogy tasks.  
Both cosine similarity and euclidean distance make it possible to compare vector representations, while simple vector arithmetic can recover country–capital relations with relatively high accuracy.